# Force and Motion


In [1]:
import sympy as sp
from sympy.physics.mechanics import *
import sympy.physics.vector as spv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
from sympy.physics.vector.printing import init_vprinting

init_vprinting(use_latex="mathjax")

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> ReferenceFrame:
    return ReferenceFrame(
        frame,
        latexs=(
            rf"\; {{}}^\mathcal {frame} \hat {x}",
            rf"\;{{}}^\mathcal {frame} \hat {y}",
            rf"\: {{}}^\mathcal {frame} \hat {z}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta"):
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

In [3]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_circular("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)

r = sp.Symbol("r", positive=True)
vec_r = r * P.x
vec_theta = P.y
assert vec_r.dot(vec_theta) == 0

PN_values = {P.x: P.x.express(N), P.y: P.y.express(N)}
display(
    Math(
        r"\boxed{"
        + r" \text{Polar to Cartesian: } \\[5pt] \\"
        + r"\begin{aligned}"
        + sp.latex(P.x)
        + " &= "
        + sp.latex(PN_values[P.x])
        + r" \\ "
        + sp.latex(P.y)
        + " &= "
        + sp.latex(PN_values[P.y])
        + r"\end{aligned}"
        + r"}"
    )
)

# Velocity
vec_v = vec_r.dt(N)
vec_a = vec_v.dt(N)
display(
    Math(
        r" \text{Velocity and Acceleration in Polar Coordinates: } \\[5pt] \\"
        + r"\boxed{"
        + r"\begin{aligned}"
        + r"\vec{v}"
        + " &= "
        + spv.vlatex(vec_v)
        + r" \\ "
        + r"\vec{a}"
        + " &= "
        + spv.vlatex(vec_a)
        + r"\end{aligned}"
        + r"}"
    )
)

# Constaint angular velocity omega, angular acceleration 0
omega = sp.Symbol("omega", positive=True)

values = {theta.diff(t): omega, theta.diff(t, 2): 0}
display(
    Math(
        r"\text{Uniform Circular Motion:} \\[5pt] \\"
        + r"\boxed{"
        + r"\begin{aligned}"
        + r"\vec{v} &= "
        + spv.vlatex(vec_v.subs(values))
        + r" && \text{where } \omega = "
        + spv.vlatex(theta.diff(t))
        + r" \text{ is constant} \\ "
        + r"\vec{a} &= "
        + spv.vlatex(vec_a.subs(values))
        + r" && \text{where } "
        + spv.vlatex(theta.diff(t, 2))
        + r" = 0"
        + r" \\ "
        + r"\end{aligned}"
        + r"}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Show that for Uniform Circular Motion

$$\boxed{\frac{d \hat{\theta}}{dt} = -\frac{d \theta}{dt}\hat{r}}$$

In [4]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_circular("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)


display(
    Math(
        r"\frac{d\hat{\theta}}{dt} =" + spv.vlatex(P.y.diff(t, N).express(P).simplify())
    )
)

<IPython.core.display.Math object>

## 11.2 WE - Car on a Banked Turn

A car of mass $m$ is turning on a banked curve of angle $\phi$ with respect to the horizontal. The curve is icy and friction between the tires and the surface is negligible. The curve has a radius $r$. What is the speed $v$ at which the car can turn safely? Express your answer in terms of $m$, $\phi$, and $r$.

In [5]:
m, phi, r, g = sp.symbols("m phi r g", positive=True)
m, phi, r, g

t = dynamicsymbols._t
theta = dynamicsymbols("theta")

N = reference_frame("N")

P = reference_frame_circular("P")
P.orient_axis(N, theta, N.z)

NmB = sp.Symbol("N_mB", positive=True)

vec_FmE = m * g * (-P.y)
vec_NmB = NmB * (sp.cos(sp.pi / 2 + phi) * P.x + sp.cos(phi) * P.y)
vec_F_total = vec_FmE + vec_NmB

vec_r = r * P.x
vec_v = vec_r.diff(t, N)
vec_a = vec_v.diff(t, N).subs(theta.diff(t, 2), 0)

display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"\vec{r} &= "
        + spv.vlatex(vec_r)
        + r"\\"
        + r"\vec{v} &= "
        + spv.vlatex(vec_v)
        + r"\\"
        + r"\vec{a} &= "
        + spv.vlatex(vec_a)
        + r"\;,&& \text{where } \ddot{\theta} = 0"
        + r"\end{aligned}}"
    )
)

N2Leq = sp.Eq(vec_F_total.to_matrix(N), m * vec_a.to_matrix(N))
result = sp.solve(N2Leq, [NmB, theta.diff(t)], dict=True)

result_vec_v = [
    vec_v.subs(_).subs(theta.diff(t, 2), 0).magnitude().simplify() for _ in result
]
display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"|\vec{v}| &= "
        + spv.vlatex(result_vec_v[0].simplify())
        + r"\end{aligned}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## PS.3.1 WE - Orbital Circular Motion

A person on a spherical asteroid of mass $m_1$ and radius $R$, sees a small satellite of mass $m_2$ orbiting the asteroid in a circular orbit of period $T$.

Express your answers in terms of some or all of the following: $m_1$, $m_2$, $\pi$, $T$, and the universal gravitational constant $G$.

(Part a) What is the radius $r_{sat}$ of the satellite’s orbit?

(Part b) What is the magnitude of the velocity of the satellite?

Part (c) If the asteroid rotates about its axis with a period $T_{a}$, at what radius must the satellite orbit the asteroid so that the satellite appears stationary to the person on the asteroid?

Express your answer in terms of some or all of the following: $m_1$, $m_2$, $\pi$, $R$, $T$, and the universal gravitational constant $G$.

Express your answer in terms of some or all of the following: $m_1$, $m_2$, $R$, $T$, and the universal gravitational constant $G$.



In [6]:
# (Part a) What is the radius $r_{sat}$ of the satellite’s orbit?

m1, m2, R, T, G, r = sp.symbols("m1 m2 R T G r", real=True, positive=True)
m1, m2, R, T, G, r

t = dynamicsymbols._t
theta = dynamicsymbols("theta")
theta_d = dynamicsymbols("theta", 1)
theta_dd = dynamicsymbols("theta", 2)
display(Math(r"\frac{d\theta}{dt} = " + spv.vlatex(theta_d)))
display(Math(r"\frac{d^2\theta}{dt^2} = " + spv.vlatex(theta_dd)))

N = reference_frame("N")

P = reference_frame_circular("P")
P.orient_axis(N, theta, N.z)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
vec_F21 = G * m1 * m2 / r ** sp.Integer(2) * (-P.x)
display(Math(r"\vec{F}_{21} = " + spv.vlatex(vec_F21)))
vec_F21_N = vec_F21.express(N)

vec_r = r * P.x
display(Math(r"\vec{r} = " + spv.vlatex(vec_r)))
vec_v = msubs(vec_r.dt(N), {theta_d: 2 * sp.pi / T})
vec_a = msubs(vec_v.dt(N), {theta_d: 2 * sp.pi / T}, {theta_dd: 0})

display(Math(r"\vec{v} = " + spv.vlatex(vec_v)))
display(Math(r"\vec{a} = " + spv.vlatex(vec_a)))

N2Leq = sp.Eq((m2 * vec_a.to_matrix(N)), vec_F21_N.to_matrix(N))
display(Math(r"\text{Newton's 2nd Law: } " + spv.vlatex(N2Leq)))


result = sp.solve(N2Leq, [r], dict=True)[0][r]

display(Math(r"\text{Solution for } r_{sat}: " + spv.vlatex(result)))
display(
    Math(
        r"\boxed{\text{Compact form: } r_{sat} = "
        + r"\sqrt[3]{\frac{G m_1 T^2}{4 \pi^2}}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
# (Part b) What is the magnitude of the velocity of the satellite?

display(Math(f"v = " + spv.vlatex(vec_v.magnitude().subs(r, result).simplify())))
display(Math(r"\text{Compact form: v = }" + r"\sqrt[3]{\frac{2 \pi m_1 G}{T}}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem Set 3

[Problem Set 3](./Problem%20Set%203.pdf)

## Problem 3.1

![Problem 3.1](../figures/PS0301-RotatingHoop.jpg)

In [9]:
display(Markdown("## Part (a)"))

# Define the symbols for mass, radius, and angular velocity
m, R, Omega = sp.symbols("m R Omega", real=True, positive=True)

# Define the dynamicsymbol for time
t = dynamicsymbols._t

# Define the dynamicsymbol for angular velocity, its first and second derivatives
omega = dynamicsymbols("omega")
omega_d = dynamicsymbols("omega", 1)
omega_dd = dynamicsymbols("omega", 2)

# Define the dynamicsymbol for angular position, its first and second derivatives
theta = dynamicsymbols("theta")
theta_d = dynamicsymbols("theta", 1)
theta_dd = dynamicsymbols("theta", 2)

# Create reference frames

# Reference frame 'N' with no transformations
N = reference_frame("N")

# Reference frame 'P' with rotational motion about the y-axis
P = reference_frame(
    "P",
    x=r"\omega",  # Angular velocity
    y=r"e_y",  # Unit vector in the y-direction
    z=r"r",  # Position vector
)

# Orient the frame with respect to the axis of rotation
P.orient_axis(N, omega, N.y)

# Reference frame 'Q' with rotational motion about the z-axis
Q = reference_frame(
    "Q",
    x=r"r",  # Position vector
    y=r"\theta",  # Angular position
    z=r"e_z",  # Unit vector in the z-direction
)

# Orient the frame with respect to the axis of rotation
Q.orient_axis(P, 3 * sp.pi / 2 + theta, P.z)

# Define the values of angular velocity and its derivatives
values_omega = {omega_d: Omega, omega_dd: 0}
values_theta = {theta_d: 0, theta_dd: 0}

## Part (a)

In [10]:
display(Markdown("#### Position, Velocity, and Acceleration"))

vec_r = R * Q.x  # Vector from the rotation axis to the particle
vec_v = (
    vec_r.dt(N).subs(values_omega).subs(values_theta)
)  # Velocity in the rotating frame
vec_a = (
    vec_v.dt(N).subs(values_omega).subs(values_theta)
)  # Acceleration in the rotating frame
display(Math(r"{{}}^\mathcal {Q}\vec{r} = " + spv.vlatex(vec_r)))
display(Math(r"{{}}^\mathcal {N}\vec{v} = " + spv.vlatex(vec_v)))
display(Math(r"{{}}^\mathcal {N}\vec{a} = " + spv.vlatex(vec_a)))

display(Markdown("---"))
display(Markdown("#### Forces on bead"))

# Force due to Earth's gravity
F_1E = m * g * (-N.y)

# Normal Force on bead by hoop
NmH = sp.Symbol(r"N_{mH}", real=True, positive=True)  # Normal force from the hoop
F_Mh = NmH * (-Q.x).express(N)  # Force on bead

# Total force
F_total = (F_1E + F_Mh).simplify()

display(Math(r"{{}}^\mathcal {N}\vec{F}_{1E} = " + spv.vlatex(F_1E)))
display(Math(r"{{}}^\mathcal {N}\vec{F}_{mH} = " + spv.vlatex(F_Mh)))
display(Math(r"{{}}^\mathcal {N}\vec{F}_{total} = " + spv.vlatex(F_total)))

#### Position, Velocity, and Acceleration

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---

#### Forces on bead

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
eqn1 = sp.Eq((F_total - m * vec_a).to_matrix(N).simplify(), sp.zeros(3, 1))
display(Markdown("#### Newton's Second Law:"))
display(eqn1)

#### Newton's Second Law:

⎡⎛           2    ⎞              ⎤      
⎢⎝-N_{mH} + Ω ⋅R⋅m⎠⋅sin(θ)⋅cos(ω)⎥   ⎡0⎤
⎢                                ⎥   ⎢ ⎥
⎢      N_{mH}⋅cos(θ) - g⋅m       ⎥ = ⎢0⎥
⎢                                ⎥   ⎢ ⎥
⎢⎛          2    ⎞               ⎥   ⎣0⎦
⎣⎝N_{mH} - Ω ⋅R⋅m⎠⋅sin(ω)⋅sin(θ) ⎦      

In [12]:
result_theta = sp.solve(eqn1, [NmH, theta], dict=True)
display(Markdown(rf"#### {len(result_theta)} solutions for $\theta$ and ${NmH}$:"))
display(result_theta)

display(Markdown("---"))
result_Omega = sp.solve(eqn1, [NmH, Omega], dict=True)
display(Markdown(rf"#### {len(result_Omega)} solutions for $\Omega$ and ${NmH}$:"))
display(Math(rf"\boxed{{{sp.latex(result_Omega)}}}"))

#### 3 solutions for $\theta$ and $N_{mH}$:

⎡                     ⎧         2               ⎛ g  ⎞      ⎫  ⎧         2     ↪
⎢{N_{mH}: g⋅m, θ: 0}, ⎪N_{mH}: Ω ⋅R⋅m, θ: - acos⎜────⎟ + 2⋅π⎪, ⎪N_{mH}: Ω ⋅R⋅m ↪
⎢                     ⎨                         ⎜ 2  ⎟      ⎬  ⎨               ↪
⎢                     ⎪                         ⎝Ω ⋅R⎠      ⎪  ⎪               ↪
⎣                     ⎩                                     ⎭  ⎩               ↪

↪          ⎛ g  ⎞⎫⎤
↪ , θ: acos⎜────⎟⎪⎥
↪          ⎜ 2  ⎟⎬⎥
↪          ⎝Ω ⋅R⎠⎪⎥
↪                ⎭⎦

---

#### 2 solutions for $\Omega$ and $N_{mH}$:

<IPython.core.display.Math object>

## Problem 3.2

![Problem 3.2](../figures/PS0302-BankedCar.jpg)

A car of mass $m$ is going around a circular turn of radius $R$, which is 
banked at an angle $\beta$ with respect to the ground. Assume there is friction 
between the wheels and the road. Let $\mu_s$ be the coefficient of static friction 
and $g$ the magnitude of the gravitational acceleration. You may neglect 
kinetic friction (that is, the car's tires do not slip). Derive an expression 
for the range of possible speeds $v_{min}$ to $v_{max}$ necessary to keep
the car moving in a circle without slipping up or down the embanked turn. 

Express your answer in terms of some or all of the following: 
$\mu_s$, $\beta$, $m$, $R$ and $g$.

In [13]:
# Define symbols for m, R, beta, mu_s, g
m, R, beta, mu_s, Omega, g = sp.symbols(
    "m R beta mu_s Omega g", positive=True, real=True
)
omega = dynamicsymbols("omega")
omega_d = dynamicsymbols("omega", 1)
omega_dd = dynamicsymbols("omega", 2)

# Define the dynamicsymbol for time
t = dynamicsymbols._t

# Reference frame 'N' with no transformations
N = reference_frame("N")

# Reference frame 'P' with rotational motion about the y-axis
P = reference_frame(
    "P",
    x=r"r",
    y=r"\theta",
    z=r"k",
)

# Orient the frame with respect to the axis of rotation
P.orient_axis(N, omega_d * t, N.y)

values_omega = {omega_d: Omega, omega_dd: 0}

In [14]:
vec_R = msubs(R * P.x, values_omega).express(N)
vec_v = msubs(vec_R.dt(N), values_omega)
vec_a = msubs(vec_v.dt(N), values_omega)

In [15]:
N_ct = sp.symbols(r"N_{ct}", positive=True, real=True)

vec_n_normal = (P.x * sp.cos(sp.pi / 2 + beta) + P.z * sp.cos(beta)).express(N)
vec_n_parallel = (P.x * sp.cos(beta) + P.z * sp.cos(sp.pi / 2 - beta)).express(N)
assert vec_n_normal.dot(vec_n_parallel).simplify() == 0, "Vectors are not perpendicular"

F_cE = m * g * (-N.z)
F_ct = (N_ct * vec_n_normal).express(N)

# No friction force mu_s = 0
# friction = r"\text{No friction:}"
F_cmu = (N_ct * mu_s * vec_n_parallel).express(N)
F_cmu = F_cmu.subs(mu_s, 0)

# Car slides down too slow so friction force is up the ramp
# friction = r"\text{Friction up ramp car sliding down too slow:}"
# F_cmu = (N_ct * mu_s * vec_n_parallel).express(N)

# Car slides up too fast so friction force is down the ramp
friction = r"\text{Friction down ramp car too fast:}"
F_cmu = (N_ct * mu_s * (-vec_n_parallel)).express(N)

# Total force
F_net = (F_cE + F_ct + F_cmu).express(N).subs(values_omega).simplify()
display(Math(rf"\Large{friction}"))
display(Math(r"\quad \vec{F}_{net} = " + sp.latex(F_net)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [16]:
eqn1 = (
    msubs((F_net - m * vec_a), values_omega).to_matrix(N).subs(values_omega).simplify()
)
eqnN2L = sp.Eq(eqn1, sp.zeros(3, 1)).subs(t, 0).simplify()

In [17]:
result = sp.solve(eqnN2L, [N_ct, Omega], dict=True)[1]
result[Omega] = sp.simplify(result[Omega])

display(Math(rf"\Large{friction}"))

display(
    Math(
        rf"\Omega = {spv.vlatex(result[Omega])}"
        + (rf", \quad N = {spv.vlatex(result[N_ct])}")
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
display(Math(rf"\Large{friction}"))

display(
    Math(
        r"\text{constant angular velocity, \; }"
        + r"\Omega = "
        + spv.vlatex(vec_v.express(P).subs(result).subs(t, 0).simplify())
    )
)

display(
    Math(
        r"\text{velocity of car at t=0, \; }"
        + r"\vec{v} = "
        + spv.vlatex(vec_v.express(P).subs(result).subs(t, 0).simplify())
    )
)

display(
    Math(
        r"\text{speed of car at } t=0, \; |\vec{v}| = "
        + spv.vlatex(vec_v.express(P).subs(result).subs(t, 0).magnitude().simplify())
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 3.3 Tetherball

A small ball of mass $m$ is suspended by a string of length $l$. The string 
makes an angle $\beta$ with the vertical. The ball revolves in a circle 
with an unknown constant angular speed $\omega$. The orbital plane of the 
ball is at a height $h$ above the ground. Let $g$ be the gravitational constant. 
You may ignore air resistance and the size of the ball.

![Problem 3.3](../figures/PS0303-Tetherball.jpg)

(a) Find an expression for the angular speed $\omega$. Express you answer 
in terms of some or all of the following: $\ell$, $\beta$, and $g$.

(b) Later, the ball detaches from the string just as it passes the x-axis. 
It flies through the air and hits the ground at an unknown horizontal 
distance $d$ from the point at which it detached from the string.

![Problem 3.3](../figures/PS0303-Tetherball01.jpg)

What horizontal distance $d$ does the ball traverse before it hits the ground? 
Express your answer in terms of some or all of the following: $\ell$, $h$, and $g$.


In [19]:
r, Omega, beta, ell, m, h, g = sp.symbols(
    "r Omega beta ell m h g", real=True, positive=True
)
omega = dynamicsymbols("omega")
omega_d = dynamicsymbols("omega", 1)
omega_dd = dynamicsymbols("omega", 2)

# Define the dynamicsymbol for time
t = dynamicsymbols._t

# Reference frame 'N' with no transformations
N = reference_frame("N")

# Reference frame 'P' with rotational motion about the y-axis
P = reference_frame(
    "P",
    x=r"r",
    y=r"\omega",
    z=r"k",
)

# Orient the frame with respect to the axis of rotation
P.orient_axis(N, omega_d * t, N.z)

values_omega = {r: ell * sp.sin(beta), omega_d: Omega, omega_dd: 0}
values_omega

{r: ell⋅sin(β), ω̇: Ω, ω̈: 0}

In [20]:
vec_r = msubs((h * P.z + r * P.x).express(N), values_omega)
display(Math(r"\vec{r} = " + spv.vlatex(vec_r)))
vec_v = msubs(vec_r.dt(N))
display(Math(r"\vec{v} = " + spv.vlatex(vec_v)))
vec_a = msubs(vec_v.dt(N))
display(Math(r"\vec{a} = " + spv.vlatex(vec_a)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [21]:
vec_ell = P.x * sp.cos(sp.pi / 2 + beta) + P.z * sp.cos(beta)
display(Math(r"{\hat{\ell}} = " + spv.vlatex(vec_ell)))
assert vec_ell.magnitude().simplify() == 1, "vec_ell should be a unit vector"

F_mE = m * g * (-P.z)
display(Math(r"{\vec{F}_{mE}} = " + spv.vlatex(F_mE)))

# Tension in string
T = sp.Symbol("T", real=True, positive=True)
vec_T = T * vec_ell
display(Math(r"{\vec{T}} = " + spv.vlatex(vec_T)))

# Total force
F_net = F_mE + vec_T
display(Math(r"{\vec{F}_{net}} = " + spv.vlatex(F_net)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [22]:
# Newton's 2nd Law: F_net - m*a = 0
eqn1 = msubs(F_net - m * vec_a, values_omega).to_matrix(N)
# Set equation equal to zero vector at t=0
eqnN2L = sp.Eq(eqn1, sp.zeros(3, 1)).subs(t, 0)
# Solve for tension T and angular speed Omega
result = sp.solve(eqnN2L, [T, Omega], dict=True)[1]

# Display result for angular speed
display(Math(r"\Large\text{Part(a) Angular speed: }"))
display(Math(r"\Omega = " + spv.vlatex(result[Omega])))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [23]:
vec_velocity_t0 = msubs(vec_v, values_omega, result).express(N).subs(t, 0)
speed_t0 = vec_velocity_t0.magnitude()

display(Markdown("#### Velocity at t=0:"))
display(
    Math(
        r"\vec{v} = "
        + spv.vlatex(vec_velocity_t0)
        + r"\\"
        + r"|\vec{v}| = "
        + spv.vlatex(speed_t0)
    )
)

#### Velocity at t=0:

<IPython.core.display.Math object>

In [24]:
# Define symbol for time under gravity
t_g = sp.symbols("t_g", real=True, positive=True)  # [s]

# Define equation of motion under gravity
eqn_g = sp.Eq(h - sp.Rational(1, 2) * g * t_g**2, 0)  # [m]

# Solve for time under gravity
result_g = sp.solve(eqn_g, t_g, dict=True)[0]  # [s]

# Calculate distance traveled under gravity
distance_d = (
    result_g[t_g]
    * speed_t0.subs(sp.Abs(sp.sin(beta)), sp.sin(beta)).simplify()
)  # [m]

display(Markdown(f"#### Part(b) Distance travelled under gravity: "))
display(Math(rf"{spv.vlatex(distance_d)}"))


#### Part(b) Distance travelled under gravity: 

<IPython.core.display.Math object>